In [0]:
# ============================================================
# Ajay | Notebook 1: API → Raw File
# ============================================================

# Read parameters from the Job (or use these defaults when run interactively)
dbutils.widgets.text("catalog", "dev_team")
dbutils.widgets.text("schema",  "testing")

CATALOG   = dbutils.widgets.get("catalog")
SCHEMA    = dbutils.widgets.get("schema")
VOLUME    = "files_volume"
BASE_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/Ajay/ajay_flights_vol"
API_URL   = "https://opensky-network.org/api/states/all"

print(f"✅ Config loaded")
print(f"   Catalog  : {CATALOG}")
print(f"   Schema   : {SCHEMA}")
print(f"   Base Path: {BASE_PATH}")

✅ Config loaded
   Catalog  : dev_team
   Schema   : testing
   Base Path: /Volumes/dev_team/testing/files_volume/Ajay/ajay_flights_vol


In [0]:
# Create the directory in your Volume
dbutils.fs.mkdirs(BASE_PATH)
print(f"✅ Directory created successfully")
print(f"   Path: {BASE_PATH}")

✅ Directory created successfully
   Path: /Volumes/dev_team/testing/files_volume/Ajay/ajay_flights_vol


In [0]:
import requests
import json
from datetime import datetime, timezone

print("🌐 Connecting to OpenSky Network API...")

try:
    response = requests.get(
        API_URL,
        timeout=30,
        headers={"Accept": "application/json"}
    )
    response.raise_for_status()
    raw_data = response.json()
    state_count = len(raw_data.get("states", [])) if raw_data.get("states") else 0
    print(f"✅ API call successful! Aircraft: {state_count}")
    API_SUCCESS = True

except Exception as e:
    print(f"⚠️ API call failed: {e}")
    print(f"🔄 Falling back to latest existing file...")
    API_SUCCESS = False

🌐 Connecting to OpenSky Network API...
✅ API call successful! Aircraft: 10497


In [0]:
if API_SUCCESS:
    now       = datetime.now(timezone.utc)
    ts_str    = now.strftime("%Y%m%d_%H%M%S")
    file_name = f"ajay_flights_{ts_str}.json"
    file_path = f"{BASE_PATH}/{file_name}"
    with open(file_path, "w") as f:
        json.dump(raw_data, f)
    print(f"✅ New file saved: {file_path}")
else:
    files = dbutils.fs.ls(BASE_PATH)
    json_files = sorted(
        [f for f in files if f.name.endswith(".json")],
        key=lambda x: x.name,
        reverse=True
    )
    if not json_files:
        raise Exception("❌ No existing files and API failed!")
    file_path = json_files[0].path
    file_name = json_files[0].name
    print(f"✅ Using existing file: {file_name}")

dbutils.jobs.taskValues.set(key="raw_file_path", value=file_path)
print(f"📁 File path: {file_path}")

✅ New file saved: /Volumes/dev_team/testing/files_volume/Ajay/ajay_flights_vol/ajay_flights_20260506_132301.json
📁 File path: /Volumes/dev_team/testing/files_volume/Ajay/ajay_flights_vol/ajay_flights_20260506_132301.json


In [0]:
# Generate timestamped filename
now       = datetime.now(timezone.utc)
ts_str    = now.strftime("%Y%m%d_%H%M%S")
file_name = f"ajay_flights_{ts_str}.json"
file_path = f"{BASE_PATH}/{file_name}"

# ✅ Write directly to Volume path (Unity Catalog compatible)
with open(file_path, "w") as f:
    json.dump(raw_data, f)

print(f"✅ File saved to Volume!")
print(f"   📄 File name  : {file_name}")
print(f"   📁 Full path  : {file_path}")
print(f"   ✈️  Records    : {state_count}")

# Pass file path to next notebook via Job task values
dbutils.jobs.taskValues.set(key="raw_file_path", value=file_path)
dbutils.jobs.taskValues.set(key="record_count",  value=str(state_count))

✅ File saved to Volume!
   📄 File name  : ajay_flights_20260505_174937.json
   📁 Full path  : /Volumes/dev_team/testing/files_volume/Ajay/ajay_flights_vol/ajay_flights_20260505_174937.json
   ✈️  Records    : 10676


In [0]:
# List files in the Volume to verify
files = dbutils.fs.ls(BASE_PATH)

print(f"📁 Files in Volume:")
print(f"   Path: {BASE_PATH}\n")

for f in sorted(files, key=lambda x: x.name, reverse=True):
    size_kb = round(f.size / 1024, 2)
    print(f"   ✅ {f.name}  →  {size_kb} KB")

print(f"\n🏁 Notebook 1 Complete!")
print(f"   Total files in volume : {len(files)}")

📁 Files in Volume:
   Path: /Volumes/dev_team/testing/files_volume/Ajay/ajay_flights_vol

   ✅ ajay_flights_20260505_174937.json  →  1550.07 KB

🏁 Notebook 1 Complete!
   Total files in volume : 1
